# Fine-tune CRNN on Government Document Crops

This notebook fine-tunes the existing CRNN checkpoint on 132 line crops
from real Maharashtra government documents.

**Goal:** Reduce CER on government docs from ~0.89 to below Tesseract's 0.57

**Time:** ~20 minutes on Colab T4

In [ ]:
# ── Cell 1: Mount Drive and clone repo ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
REPO = '/content/aksharDrishti'
DRIVE_WEIGHTS = '/content/drive/MyDrive/AksharDrishti/weights'

if not os.path.exists(REPO):
    !git clone https://github.com/Ayush-04-spec/akshardrishti.git {REPO}
else:
    !cd {REPO} && git pull

os.makedirs(DRIVE_WEIGHTS + '/recognize', exist_ok=True)
print('Repo ready:', REPO)

In [ ]:
# ── Cell 2: Install dependencies ────────────────────────────────────────────
!pip install torch torchvision --quiet
import torch
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', torch.cuda.get_device_properties(0).total_memory/1e9, 'GB')

In [ ]:
# ── Cell 3: Copy weights and govdoc dataset from Drive ──────────────────────
import shutil

# Copy existing CRNN weights
src_weights = DRIVE_WEIGHTS + '/recognize/crnn_mozhi_himr.pt'
dst_weights = REPO + '/weights/recognize/crnn_mozhi_himr.pt'
os.makedirs(REPO + '/weights/recognize', exist_ok=True)

if os.path.exists(src_weights):
    shutil.copy(src_weights, dst_weights)
    print('Weights copied from Drive')
else:
    print('WARNING: weights not found on Drive at', src_weights)
    print('Upload crnn_mozhi_himr.pt to', DRIVE_WEIGHTS + '/recognize/')

# Copy govdoc finetune dataset
src_data = DRIVE_WEIGHTS + '/../datasets/govdoc_finetune'
dst_data = REPO + '/datasets/govdoc_finetune'

if os.path.exists(src_data):
    if os.path.exists(dst_data):
        shutil.rmtree(dst_data)
    shutil.copytree(src_data, dst_data)
    import glob
    n_imgs = len(glob.glob(dst_data + '/images/*.png'))
    print(f'Dataset copied: {n_imgs} images')
else:
    print('Dataset not found on Drive. Upload datasets/govdoc_finetune/ to Drive first.')
    print('Expected path:', src_data)

In [ ]:
# ── Cell 4: Fix absolute paths in CSV ───────────────────────────────────────
import csv, re
from pathlib import Path

csv_path = REPO + '/datasets/govdoc_finetune/index.csv'
rows = list(csv.DictReader(open(csv_path, encoding='utf-8')))

# Rewrite Windows absolute paths to Colab paths
fixed = []
for r in rows:
    img_p = r['image_path']
    # Extract just the filename
    fname = Path(img_p).name
    r['image_path'] = REPO + '/datasets/govdoc_finetune/images/' + fname
    fixed.append(r)

with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['image_path','text','language','split'])
    w.writeheader()
    w.writerows(fixed)

print(f'Fixed {len(fixed)} rows in CSV')
print('Sample:', fixed[0])

In [ ]:
# ── Cell 5: Fine-tune ────────────────────────────────────────────────────────
import sys
sys.path.insert(0, REPO)

import torch, time, csv, unicodedata
import torch.nn as nn
from torch.utils.data import DataLoader
from pathlib import Path

# Load charset and dataset tools
from akshardrishti.recognize.crnn_model import Charset, greedy_decode
from train.dataset import LineImageDataset, ctc_collate, load_index
from akshardrishti.postprocess.text_repair import normalize_for_scoring

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

CHARSET_PATH = REPO + '/weights/recognize/charset_deva.txt'
WEIGHTS_PATH = REPO + '/weights/recognize/crnn_mozhi_himr.pt'
GOVDOC_CSV   = REPO + '/datasets/govdoc_finetune/index.csv'
OUT_PATH     = REPO + '/weights/recognize/crnn_govdoc_ft.pt'

# Load charset
charset = Charset.load(Path(CHARSET_PATH))
print(f'Charset size: {len(charset)}')

# Load base model
ckpt = torch.load(WEIGHTS_PATH, map_location='cpu', weights_only=True)
from akshardrishti.recognize.crnn_model import CRNN
model = CRNN(num_classes=len(charset)).to(DEVICE)
model.load_state_dict(ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt)
print('Base model loaded')

# Load govdoc dataset
rows = load_index(Path(GOVDOC_CSV))
# Filter by charset coverage
rows = [r for r in rows if charset.coverage(r['text']) >= 0.8]
print(f'Govdoc rows after coverage filter: {len(rows)}')

# Also load a small sample from Mozhi to prevent catastrophic forgetting
# Use 5x more Mozhi than govdoc rows
MOZHI_CSV = REPO + '/datasets/mozhi_raw/index_train.csv'
if Path(MOZHI_CSV).exists():
    mozhi_rows = load_index(Path(MOZHI_CSV))
    # Sample randomly
    import random; random.seed(42)
    n_mozhi = min(len(rows) * 5, len(mozhi_rows))
    mozhi_sample = random.sample(mozhi_rows, n_mozhi)
    mozhi_sample = [r for r in mozhi_sample if charset.coverage(r['text']) >= 0.999]
    all_rows = rows + mozhi_sample
    print(f'Total rows (govdoc + mozhi sample): {len(all_rows)}')
else:
    all_rows = rows
    print('Mozhi not available, training on govdoc only')

random.shuffle(all_rows)

# Build dataset and loader
ds = LineImageDataset(all_rows, charset, img_height=32, max_width=512, augment=True)
loader = DataLoader(ds, batch_size=32, shuffle=True, num_workers=2,
                    collate_fn=ctc_collate, pin_memory=True)
print(f'Batches per epoch: {len(loader)}')

In [ ]:
# ── Cell 6: Training loop (10 epochs) ───────────────────────────────────────

def edit_distance(a, b):
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b)+1))
    for ca in a:
        cur = [prev[0]+1]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j]+1, cur[j-1]+1, prev[j-1]+(ca!=cb)))
        prev = cur
    return prev[-1]

def corpus_cer(preds, refs):
    err = chars = 0
    for p, r in zip(preds, refs):
        pn = normalize_for_scoring(p); rn = normalize_for_scoring(r)
        err += edit_distance(pn, rn); chars += len(rn)
    return err/chars if chars else 0

# Use lower LR for fine-tuning to avoid destroying Mozhi performance
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
ctc_loss  = nn.CTCLoss(blank=charset.blank_idx, zero_infinity=True)

EPOCHS = 10
best_cer = float('inf')

for epoch in range(1, EPOCHS+1):
    model.train()
    total_loss = 0
    t0 = time.perf_counter()

    for images, targets, input_lengths, target_lengths, _ in loader:
        images = images.to(DEVICE)
        targets = targets.to(DEVICE)

        log_probs = model(images)
        T, N, C = log_probs.shape
        loss = ctc_loss(
            log_probs,
            targets,
            torch.full((N,), T, dtype=torch.long),
            target_lengths.to(DEVICE),
        )
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()
    avg_loss = total_loss / len(loader)
    elapsed = time.perf_counter() - t0

    # Quick CER on govdoc rows only
    model.eval()
    govdoc_ds = LineImageDataset(rows, charset, img_height=32, max_width=512, augment=False)
    govdoc_loader = DataLoader(govdoc_ds, batch_size=64, shuffle=False,
                               num_workers=2, collate_fn=ctc_collate)
    preds, refs_list = [], [r['text'] for r in rows]
    with torch.no_grad():
        for images, _, _, _, texts in govdoc_loader:
            lp = model(images.to(DEVICE))
            preds.extend(t for t, _ in greedy_decode(lp, charset))

    cer = corpus_cer(preds, refs_list)
    print(f'Epoch {epoch:2d}/{EPOCHS}  loss={avg_loss:.4f}  govdoc_CER={cer:.4f}  ({elapsed:.0f}s)')

    if cer < best_cer:
        best_cer = cer
        torch.save({'model_state_dict': model.state_dict(),
                    'charset_size': len(charset),
                    'epoch': epoch, 'best_cer': best_cer},
                   OUT_PATH)
        print(f'  -> New best CER {best_cer:.4f}, saved to {OUT_PATH}')

print(f'\nFine-tuning complete. Best govdoc CER: {best_cer:.4f}')

In [ ]:
# ── Cell 7: Save to Drive ────────────────────────────────────────────────────
import shutil

dst = DRIVE_WEIGHTS + '/recognize/crnn_govdoc_ft.pt'
shutil.copy(OUT_PATH, dst)
print(f'Saved fine-tuned weights to Drive: {dst}')
print(f'Best govdoc CER: {best_cer:.4f}')
print(f'Download and place at: weights/recognize/crnn_govdoc_ft.pt')